# Chunking + retrieval (F1.3 + F1.4)

See how a 100+ page filing becomes searchable: split into chunks, embed each chunk into a vector, store in FAISS, then find the nearest chunks to a question.

**How to run:** VS Code with the `.venv` kernel, or `uv run jupyter lab`.

> The first time, build the index with `uv run python -m policy_copilot.index` (downloads the embedding model). This notebook then loads it instantly.

In [ ]:
from policy_copilot.chunking import chunk_filings, chunk_text
from policy_copilot.financebench import build_corpus, load_rows, select_documents
from policy_copilot.index import INDEX_DIR, build_index, load_index, save_index, search

rows = load_rows()
corpus = build_corpus(rows, select_documents(rows, n=8))
[f.doc_id for f in corpus]

## 1. Chunk one filing

The AMD 10-K (121 pages) becomes many small, paragraph-aligned chunks.

In [ ]:
amd = next(f for f in corpus if f.doc_id == "AMD_2022_10K")
amd_chunks = chunk_text(amd.doc_id, amd.text)
print(f"{amd.n_pages} pages -> {len(amd_chunks)} chunks")

In [ ]:
c = amd_chunks[150]
print("chunk_id:", c.chunk_id)
print("heading :", c.heading)
print("text    :", c.text[:400], "...")

## 2. Chunk the whole corpus + build / load the index

In [ ]:
chunks = chunk_filings(corpus)
print(f"{len(corpus)} filings -> {len(chunks)} chunks")

if (INDEX_DIR / "faiss.index").exists():
    index, chunks = load_index()
else:
    index = build_index(chunks)
    save_index(index, chunks)

print("vectors in index:", index.ntotal)

## 3. Search — in-corpus vs out-of-corpus

Watch the similarity scores: a question our filings can answer scores high; one they cannot scores low. That gap is what drives the refusal behaviour later.

In [ ]:
def show(query, k=3):
    print("Q:", query)
    for hit in search(index, chunks, query, k):
        print(f"  {hit.score:.3f}  [{hit.chunk.chunk_id}]  {hit.chunk.heading[:40]}")
        print("        ", " ".join(hit.chunk.text.split())[:110])
    print()


show("What was AMD's total net revenue in fiscal 2022?")  # in corpus
show("What is the capital of France?")  # out of corpus -> low scores

## 4. Try your own

Edit the query below — try a Boeing or PepsiCo question, or something the filings can't answer.

In [ ]:
show("How much did Boeing spend on research and development?")